In [1]:
import csv
import io
import time
from datetime import datetime, timezone

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# ---------------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------------
KUSTO_QUERY_URI = "https://trd-s0c8mwesu3tzwvrv46.z0.kusto.fabric.microsoft.com"
KUSTO_DATABASE = "NYC_Weather_Eventhouse"
KUSTO_TABLE = "nyc_weather_hourly"

START_DATE, END_DATE = "2024-01-01", "2024-03-31"   # aligned with NYC Taxi Q1 2024
LAT, LON = 40.7128, -74.0060
LOCAL_TZ = "America/New_York"
BATCH_SIZE = 500

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Target schema (column order == CSV order for .ingest inline)
KQL_COLUMNS = [
    ("timestamp_utc", "datetime"),
    ("timestamp_local", "datetime"),
    ("city", "string"),
    ("latitude", "real"),
    ("longitude", "real"),
    ("temperature_c", "real"),
    ("precipitation_mm", "real"),
    ("rain_mm", "real"),
    ("snowfall_cm", "real"),
    ("wind_speed_kmh", "real"),
    ("is_rainy_hour", "int"),
    ("temp_band", "string"),
    ("taxi_trips", "long"),
    ("taxi_avg_fare_usd", "real"),
    ("taxi_revenue_usd", "real"),
    ("taxi_avg_duration_min", "real"),
    ("pm25_daily_avg_ugm3", "real"),
    ("ingested_at_utc", "datetime"),
]

# ---------------------------------------------------------------------------
# 2. Fetch hourly weather from Open-Meteo (in UTC) with retry/back-off
# ---------------------------------------------------------------------------
params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": "temperature_2m,precipitation,rain,snowfall,wind_speed_10m",
    "timezone": "GMT",
}

hourly = None
MAX_ATTEMPTS = 4
for attempt in range(1, MAX_ATTEMPTS + 1):
    try:
        resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=90)
        resp.raise_for_status()
        hourly = resp.json()["hourly"]
        break
    except (requests.exceptions.RequestException, KeyError, ValueError) as e:
        if attempt == MAX_ATTEMPTS:
            raise RuntimeError(f"Open-Meteo request failed after {MAX_ATTEMPTS} attempts: {e}") from e
        wait = 5 * attempt
        print(f"Attempt {attempt} failed ({type(e).__name__}: {e}); retrying in {wait}s...")
        time.sleep(wait)

def to_float(v):
    return None if v is None else float(v)

raw_rows = [
    (t, to_float(temp), to_float(prec), to_float(rain), to_float(snow), to_float(wind))
    for t, temp, prec, rain, snow, wind in zip(
        hourly["time"], hourly["temperature_2m"], hourly["precipitation"],
        hourly["rain"], hourly["snowfall"], hourly["wind_speed_10m"],
    )
]

raw_schema = StructType([
    StructField("time_utc", StringType(), False),
    StructField("temperature_c", DoubleType(), True),
    StructField("precipitation_mm", DoubleType(), True),
    StructField("rain_mm", DoubleType(), True),
    StructField("snowfall_cm", DoubleType(), True),
    StructField("wind_speed_kmh", DoubleType(), True),
])

df_weather = (
    spark.createDataFrame(raw_rows, schema=raw_schema)
    .withColumn("ts_utc", F.to_timestamp("time_utc", "yyyy-MM-dd'T'HH:mm"))
    .withColumn("ts_local", F.from_utc_timestamp("ts_utc", LOCAL_TZ))
    .withColumn("local_date", F.to_date("ts_local"))
    .withColumn("local_hour", F.hour("ts_local"))
    .withColumn("is_rainy_hour", (F.coalesce(F.col("precipitation_mm"), F.lit(0.0)) >= 0.5).cast("int"))
    .withColumn(
        "temp_band",
        F.when(F.col("temperature_c") < 0, "freezing")
         .when(F.col("temperature_c") < 10, "cold")
         .when(F.col("temperature_c") < 20, "mild")
         .otherwise("warm"),
    )
)

# ---------------------------------------------------------------------------
# 3. Enrich with taxi demand (hourly) and air quality (daily PM2.5) from Silver
# ---------------------------------------------------------------------------
def read_silver(table_name):
    try:
        return spark.table(table_name)
    except Exception as e:
        raise RuntimeError(
            f"Cannot read '{table_name}'. Attach Lakehouse_Silver as the default lakehouse of this notebook."
        ) from e

taxi_hourly = (
    read_silver("nyc_taxi_silver")
    .filter(F.col("pickup_date").between(START_DATE, END_DATE))
    .groupBy(F.col("pickup_date").alias("local_date"), F.col("pickup_hour").alias("local_hour"))
    .agg(
        F.count("*").alias("taxi_trips"),
        F.round(F.avg("fare_amount"), 2).alias("taxi_avg_fare_usd"),
        F.round(F.sum("total_amount"), 2).alias("taxi_revenue_usd"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("taxi_avg_duration_min"),
    )
)

pm25_daily = (
    read_silver("openaq_measurements_silver")
    .filter(F.col("parameter_name") == "pm25")
    .select(
        F.col("measurement_date").alias("local_date"),
        F.col("daily_avg_value").cast("double").alias("pm25_daily_avg_ugm3"),
    )
    .dropDuplicates(["local_date"])
)

ingested_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

df_final = (
    df_weather
    .join(taxi_hourly, ["local_date", "local_hour"], "left")
    .join(pm25_daily, ["local_date"], "left")
    .select(
        F.date_format("ts_utc", "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("timestamp_utc"),
        F.date_format("ts_local", "yyyy-MM-dd'T'HH:mm:ss").alias("timestamp_local"),
        F.lit("New York").alias("city"),
        F.lit(LAT).alias("latitude"),
        F.lit(LON).alias("longitude"),
        "temperature_c", "precipitation_mm", "rain_mm", "snowfall_cm", "wind_speed_kmh",
        "is_rainy_hour", "temp_band",
        "taxi_trips", "taxi_avg_fare_usd", "taxi_revenue_usd", "taxi_avg_duration_min",
        "pm25_daily_avg_ugm3",
        F.lit(ingested_at).alias("ingested_at_utc"),
    )
    .orderBy("timestamp_utc")
)

rows = df_final.collect()
row_count = len(rows)
print(f"Prepared {row_count} enriched hourly rows "
      f"({sum(1 for r in rows if r['taxi_trips'] is not None)} with taxi data).")

# ---------------------------------------------------------------------------
# 4. Write to Eventhouse via KQL REST API (token of the current Fabric session)
# ---------------------------------------------------------------------------
def get_kusto_token():
    try:
        get_token = notebookutils.credentials.getToken
    except NameError:
        get_token = mssparkutils.credentials.getToken
    last_error = None
    for audience in (KUSTO_QUERY_URI, "kusto", "https://kusto.kusto.windows.net"):
        try:
            return get_token(audience)
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Could not obtain a Kusto token: {last_error}")

headers = {
    "Authorization": f"Bearer {get_kusto_token()}",
    "Content-Type": "application/json; charset=utf-8",
}

def kusto_post(endpoint, csl):
    res = requests.post(
        f"{KUSTO_QUERY_URI}/v1/rest/{endpoint}",
        headers=headers,
        json={"db": KUSTO_DATABASE, "csl": csl},
        timeout=120,
    )
    if not res.ok:
        raise RuntimeError(f"KQL {endpoint} error ({res.status_code}): {res.text[:1000]}")
    return res.json()

# Recreate the table so the column order always matches the CSV (idempotent full reload)
schema_kql = ", ".join(f"{name}: {kql_type}" for name, kql_type in KQL_COLUMNS)
kusto_post("mgmt", f".drop table {KUSTO_TABLE} ifexists")
kusto_post("mgmt", f".create table {KUSTO_TABLE} ({schema_kql})")

column_names = [name for name, _ in KQL_COLUMNS]
for start in range(0, row_count, BATCH_SIZE):
    buf = io.StringIO()
    writer = csv.writer(buf, lineterminator="\n")
    for r in rows[start:start + BATCH_SIZE]:
        writer.writerow(["" if r[c] is None else r[c] for c in column_names])
    kusto_post("mgmt", f".ingest inline into table {KUSTO_TABLE} <|\n{buf.getvalue().rstrip()}")
    print(f"Ingested rows {start + 1}-{min(start + BATCH_SIZE, row_count)}")

# ---------------------------------------------------------------------------
# 5. Verify
# ---------------------------------------------------------------------------
result = kusto_post("query", f"{KUSTO_TABLE} | count")
kql_count = result["Tables"][0]["Rows"][0][0]
print(f"Rows in {KUSTO_DATABASE}.{KUSTO_TABLE}: {kql_count} (sent: {row_count})")
if kql_count != row_count:
    raise RuntimeError("Row count mismatch between Spark and Eventhouse.")
print("Done.")

StatementMeta(, dc8e0ca0-f59f-419f-85cc-c7168cfad468, 3, Finished, Available, Finished, False)

Prepared 2184 enriched hourly rows (2179 with taxi data).
Ingested rows 1-500
Ingested rows 501-1000
Ingested rows 1001-1500
Ingested rows 1501-2000
Ingested rows 2001-2184
Rows in NYC_Weather_Eventhouse.nyc_weather_hourly: 2184 (sent: 2184)
Done.
